# Two-headed PFN (latent + noisy bar heads) on the noisy BNN prior

## Description

**Question.** Can one PFN, trained on a prior whose observation noise varies widely across functions, give
(a) a calibrated posterior over the *latent* f(x) and (b) a calibrated predictive of the *noisy* y, from
noisy contexts alone? Issue #3 (requested by anytime-acquisition#9). This is Stage 0 of the "learned
acquisition policies on a PFN surrogate" plan: every later acquisition (latent-incumbent EI, replication,
noise-aware option selection) reads these two heads.

**Setup.**
- Prior `priors/noisy_bnn.py`: the BNN family of `pfn_large`, its internal noise off. Per-input relevance
  sets the effective dimension (d_eff ~ U{1..d}, other inputs irrelevant or weakly relevant at 0.05).
  The latent ECDF is refitted for this family (packaged asset per d).
- Noise relative to each function's own range: SNR = (median f - min f) / robust noise scale, with
  log10 SNR ~ U[log10 2, 2]. Regimes: 20% noise-free, 40% homoscedastic, 40% heteroscedastic (scale ratio
  up to 10). Families: 60% Gaussian, 20% Student-t (nu in [3, 10]), 10% contaminated, 10% skew-normal.
- Contexts: half uniform, half BO-like (uniform start, then perturbations around the best *observed*
  points), 40% with exact replicates (fresh noise). Locations depend only on earlier observations.
- Model `surrogates/two_head_pfn.py`: the PFN backbone with a latent bar head (targets: f at the test
  points, privileged) and a separately trained noisy bar head (targets: fresh noisy y). Loss: the sum of
  both NLLs. Decoupled PFNs (arXiv:2605.06413) trains the latent head the same way but derives the
  observation predictive by a plug-in convolution with a noise variance head; see the log entry.
- One run per d in {2, 4, 8}; held-out validation per regime (64 functions x 200 points, n in {5, 20, 50}).
- The evaluation of the trained checkpoints is a separate experiment (`eval_noisy_pfn`, notebook 07).

**What to look for.** Training and validation NLL of both heads per regime over steps; whether noise-free
functions are still fitted as well as in a noise-free prior (no "noise everywhere" hedging), and whether
the noisy head's NLL improves with n in the noisy regimes (it can only reach the noise floor).

**GO / NO-GO** (pilot, per d): GO to the evaluation (notebook 07) if training is stable (no NaN, validation
NLL of both heads still decreasing or flat over the last 20% of steps, not rising). NO-GO if either head
diverges. The scientific gate is in notebook 07.

**Limits.**
- One seed, one size per d. Contexts up to 100 points; everything beyond is out of range.
- y is on this prior's own ECDF scale, not pfn_large's: the two are not directly comparable in NLL.
- Contaminated noise is not mean-zero given f, so there E[y | x] differs from f; the latent head still targets f.


In [ ]:
from pfn_repro.analysis import load_snapshot, paired_diff, show_header, summarize

SNAPSHOT = "results/noisy_pfn/<sweep_id>"  # set after `just snapshot runs/noisy_pfn/<sweep_id>`
snap = load_snapshot(SNAPSHOT)
show_header(snap)  # how it was run + configuration, generated from the snapshot

## Results

In [ ]:
# Aggregates use DONE members only; n_failed / n_started / n_missing say what is not in them yet.
summarize(snap.runs, "<metric>")

## Verdict

The GO / NO-GO criterion from the description, evaluated, in one line. Record it with `just decide`.